In [10]:
import requests
import pandas as pd
import numpy as np
import time
import uuid
from datetime import timedelta
import random
import os

# Attachment handling
from docx import Document
import io
import pypdf


In [2]:
def fetch_transactions(from_date, to_date): 
    # 1. Define the URL
    url = "https://api3.oslo.oslobors.no/v1/newsreader/list"
    params = {
        "category": "1102",
        "fromDate": from_date,
        "toDate": to_date
    }
    
    # 2. Make the GET request
    response = requests.get(url, params=params)
    
    # 3. Check if the request was successful
    if response.status_code == 200:
        # 4. Parse the JSON data
        return response.json()
        print("Success! Data received.")
    else:
        print(f"Failed with status: {response.status_code}")
        return pd.DataFrame()


In [3]:
def get_message_data(msg_id, num_att):
    # 1. Constrain the number of attachments we WANT to extract to max 10
    # Even if we only want 2, we still need to return 10 slots for Pandas
    extract_count = min(num_att, 10)
    
    url = f"https://api3.oslo.oslobors.no/v1/newsreader/message?messageId={msg_id}"
    
    try:
        data = requests.get(url).json().get('data', {}).get('message', {})
        body = data.get('body', "")
        
        # 2. Extract the actual data (up to the constrained limit)
        atts = [(a.get('id'), a.get('name')) for a in data.get('attachments', [])[:extract_count]]
        
        # 3. CRITICAL: Always pad to exactly 10 slots to satisfy Pandas
        # This ensures the list length is ALWAYS 11 (1 body + 10 atts)
        atts += [None] * (10 - len(atts))
        
        return [body] + atts
        
    except Exception:
        # 4. Fallback must also be length 11
        return [None] * 11



In [4]:
def get_attachment_text(url, file_name):
    response = requests.get(url)
    file_extension = file_name.split('.')[-1].lower()
    
    # Handle PDF
    if file_extension == 'pdf':
        # Wrap the bytes in a stream
        reader = pypdf.PdfReader(io.BytesIO(response.content))
    
        # Extract text from every page and join with newlines
        return "\n".join([page.extract_text() for page in reader.pages if page.extract_text()])
    
    # Handle Word
    elif file_extension in ['doc', 'docx']:
        doc = Document(io.BytesIO(response.content))
        return "\n".join([para.text for para in doc.paragraphs])
    
    # Handle Plain Text
    else:
        return response.text

In [5]:
# A list to store URLs that need manual checking
manual_review_list = []

def extract_row_attachments(row):
    results = {}
    message_id = row['messageId']
    
    for i in range(1, 11):
        col_name = f'attachment_{i}'
        target_col = f'string_{i}'
        attachment_data = row.get(col_name)
        
        if attachment_data and isinstance(attachment_data, tuple):
            try:
                attachment_id, file_name = attachment_data
                url = (f"https://api3.oslo.oslobors.no/v1/newsreader/attachment?"
                       f"messageId={message_id}&attachmentId={attachment_id}")
                
                # 1. Fetch text
                text = get_attachment_text(url, file_name)
                
                # 2. Check if it's a "Picture PDF" (Successful request but 0 text)
                if text == "": 
                    # This is a deviation! Add it to our manual list
                    manual_review_list.append({
                        'messageId': message_id,
                        'attachmentId': attachment_id,
                        'fileName': file_name,
                        'url': url
                    })
                    results[target_col] = "[MANUAL_REVIEW_REQUIRED]"
                else:
                    results[target_col] = text
                    
            except Exception as e:
                results[target_col] = f"Error: {str(e)}"
        else:
            results[target_col] = None
            
    return pd.Series(results)

In [6]:
def process_attachments(df, extraction_func):
    """
    Takes a DataFrame, applies an extraction function to each row,
    and maps the results to 10 'string_x' columns.
    """
    # 0. Add the needed columns (empty)
    new_cols = ['message_body'] + [f'attachment_{i}' for i in range(1, 11)]
    df[new_cols] = df.apply(
        lambda r: get_message_data(r['messageId'], r['numbAttachments']), 
        axis=1, 
        result_type='expand'
    )
    
    # 1. Create a copy to avoid SettingWithCopy warnings
    df_result = df.copy()

    # 2. Generate the 10 column names dynamically
    string_cols = [f'string_{i}' for i in range(1, 11)]

    # 3. Apply the extraction logic
    # The result of extraction_func should be a list/Series of 10 values
    df_result[string_cols] = df_result.apply(extraction_func, axis=1)

    return df_result

# Usage:
# df_enriched = process_attachments(df_overview, extract_row_attachments)

In [7]:
def melt_messages(df): 

    # 1. Define the columns you want to keep as identifiers
    id_vars = ['messageId', 'issuerSign', 'publishedTime']

    # 2. Define all potential columns that contain text
    value_vars = ['message_body'] + [f'string_{i}' for i in range(1, 11)]

    # 3. Transform the dataframe
    # 'melt' turns wide columns into rows
    df_long = pd.melt(
        df, 
        id_vars=id_vars, 
        value_vars=value_vars, 
        value_name='messageText'
    )

    # 4. Clean up
    # Drop the 'variable' column (which holds the original column names like 'string_1')
    # Drop rows where messageText is None or NaN
    df_final = (
        df_long
        .drop(columns='variable')
        .dropna(subset=['messageText'])
        .reset_index(drop=True)
    )
    return df_final

In [8]:
def append_to_parquet_dataset(new_df, folder_name, from_date, to_date):
    # Ensure folder exists
    os.makedirs(folder_name, exist_ok=True)
    
    # Generate a unique filename to avoid overwriting
    unique_filename = f"{folder_name}/batch_from_{from_date}_to_{to_date}.parquet"
    
    # Save the new batch
    new_df.to_parquet(unique_filename, index=False, engine='fastparquet')

In [11]:
# 0. Define overall date range
start_date_str = "2025-09-01"
end_date_str = "2025-09-08"
data_folder = "transaction_data"

# Convert to datetime objects
start_dt = pd.to_datetime(start_date_str)
end_dt = pd.to_datetime(end_date_str)

# 1. Loop through weeks
current_start = start_dt
counter = 0
while current_start < end_dt:
    current_end = current_start + timedelta(days=6)
    
    # Ensure we don't go past the final end date
    if current_end > end_dt:
        current_end = end_dt

    # Format dates for the API
    date_from = current_start.strftime('%Y-%m-%d')
    date_to = current_end.strftime('%Y-%m-%d')
    
    print(f"Processing: {date_from} to {date_to}...")

    try:
        # 2. Fetch
        data = fetch_transactions(date_from, date_to)
        
        # 3. Convert to DataFrame (with safety check)
        messages = data.get('data', {}).get('messages', [])
        if not messages:
            print(f"No messages found for {date_from}. Skipping.")
        else:
            df_overview = pd.DataFrame(messages)

        # 4. Process Attachments
        df_enriched = process_attachments(df_overview, extract_row_attachments)
        # 5. Melt
        df_message = melt_messages(df_enriched)

        # 6. Save (Now using the folder-based append logic)
        append_to_parquet_dataset(df_message, data_folder, date_from, date_to)
        
        print(f"Done with week {date_from}. Sleeping a bit...")
        
        # 7. Wait to avoid "Too many connections" error
        time.sleep(random.uniform(3, 10)) 
        counter += 1
        if (counter % 4 == 0):
            time.sleep(random.uniform(15, 20))

    except Exception as e:
        print(f"Error during week {date_from}: {e}")
        # Optional: wait longer if an error occurs to let the server recover
        time.sleep(30) 

    # Move to the next week
    current_start = current_end + timedelta(days=1)

print("Full process completed!")

Processing: 2025-09-01 to 2025-09-07...
Done with week 2025-09-01. Sleeping a bit...
Full process completed!
